# v2 · Fase 2 · paso 1 — Preparación del dataset

El dataset `dataset_v2/` ya viene en formato YOLO bbox correcto (5 tokens). Lo único que necesitamos antes de entrenar es **un `data.yaml` con paths absolutos** — el original usa `../train/images` que se resuelve relativo al CWD del proceso de Ultralytics y suele fallar.

Este notebook:
1. Verifica integridad de imágenes/labels.
2. Genera `dataset_v2/data_train.yaml` con paths absolutos.
3. Reporta el split definitivo (sin re-particionar — el de Roboflow es el que usaremos).

In [1]:
from pathlib import Path
from collections import Counter
import yaml
from PIL import Image

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "dataset_v2").exists():
    ROOT = ROOT.parent
DATASET = ROOT / "dataset_v2"
REPORTS = ROOT / "reports" / "v2"; REPORTS.mkdir(parents=True, exist_ok=True)
print(f"DATASET = {DATASET}")

DATASET = /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2


## 1. Verificación de integridad por split

In [2]:
summary = []
issues = []
for split in ["train", "valid", "test"]:
    img_dir = DATASET / split / "images"
    lbl_dir = DATASET / split / "labels"
    n_imgs = n_lbls = n_boxes = n_orphan = n_bad = 0
    cls_counts = Counter()
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
        n_imgs += 1
        # Verificar que la imagen abre
        try:
            with Image.open(img) as im:
                _ = (im.width, im.height)
        except Exception as e:
            issues.append(f"{split}/{img.name}: imagen corrupta ({e})")
        # Label correspondiente
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            n_orphan += 1
            continue
        n_lbls += 1
        for ln, raw in enumerate(lbl.read_text().splitlines(), start=1):
            t = raw.strip().split()
            if not t: continue
            if len(t) != 5:
                n_bad += 1
                issues.append(f"{split}/{lbl.name}:{ln} tiene {len(t)} tokens (esperaba 5)")
                continue
            try:
                cid = int(t[0])
                xc, yc, w, h = map(float, t[1:])
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    n_bad += 1
                    issues.append(f"{split}/{lbl.name}:{ln} fuera de rango [0,1]")
                else:
                    cls_counts[cid] += 1
                    n_boxes += 1
            except ValueError:
                n_bad += 1
                issues.append(f"{split}/{lbl.name}:{ln} tokens no numéricos")
    summary.append({
        "split": split, "images": n_imgs, "labels": n_lbls,
        "boxes": n_boxes, "img_sin_label": n_orphan, "lineas_malas": n_bad,
    })

import pandas as pd
print(pd.DataFrame(summary).set_index("split"))
print(f"\nIssues encontrados: {len(issues)}")
for i in issues[:10]:
    print("  -", i)
if len(issues) > 10:
    print(f"  … ({len(issues) - 10} más)")

       images  labels  boxes  img_sin_label  lineas_malas
split                                                    
train      31      31    352              0             0
valid       4       4     40              0             0
test        4       4     26              0             0

Issues encontrados: 0


## 2. Generación del `data_train.yaml` con paths absolutos

Ultralytics permite usar `path:` como prefijo y luego rutas relativas a ese path para `train/val/test`. Es el patrón más robusto.

In [3]:
with open(DATASET / "data.yaml") as f:
    src = yaml.safe_load(f)

raw_names = src["names"]
if isinstance(raw_names, list):
    names = raw_names
elif isinstance(raw_names, dict):
    names = [raw_names[k] for k in sorted(raw_names)]
else:
    raise ValueError(f"Formato de 'names' inesperado: {type(raw_names)}")

out = {
    "path": str(DATASET.resolve()),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(names),
    "names": names,
}

yaml_path = DATASET / "data_train.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(out, f, sort_keys=False)

print(yaml_path.read_text())

path: /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2
train: train/images
val: valid/images
test: test/images
nc: 5
names:
- '100'
- '1000'
- '200'
- '50'
- '500'

